# Step 3 - Preprocesamiento con GenericTransformer + temporal split


In [ ]:
# Imports y configuracion del proyecto
from project_config import init_notebook
config = init_notebook()

import utils
logger = utils.init_logger('info')


In [ ]:
import json
import pandas as pd
from pathlib import Path
from transformers.generic_transformer import GenericTransformer
from models.stacking.temporal_split import temporal_train_val_test_split

raw = Path(config['project_folder']) / config['data']['raw']['local_path'] / 'features.parquet'
trans_cfg = json.loads((Path(config['project_folder']) / 'src' / 'config' / 'transformations.json').read_text())


## 3.1 Cargar dataset (muestreo si no cabe en RAM)


In [ ]:
# Para datasets grandes, considera procesar por chunks o usar dask
df = pd.read_parquet(raw)
print(df.shape)


## 3.2 Temporal split


In [ ]:
train_df, val_df, test_df = temporal_train_val_test_split(df, date_col='date', config=config)
print('train:', train_df.shape, 'val:', val_df.shape, 'test:', test_df.shape)


## 3.3 Fit GenericTransformer SOLO en train (evitar leakage)


In [ ]:
target = config['model']['objective_column']
transformer = GenericTransformer(trans_cfg)
transformer.fit(train_df.drop(columns=[target]), train_df[target])


## 3.4 Transformar train/val/test y guardar


In [ ]:
processed_dir = Path(config['project_folder']) / config['data']['processed']['local_path']
processed_dir.mkdir(parents=True, exist_ok=True)

for name, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    transformed = transformer.transform(d.drop(columns=[target]))
    transformed[target] = d[target].values
    transformed.to_parquet(processed_dir / f'{name}.parquet')
    print(name, transformed.shape)


## 3.5 Guardar transformer y features map


In [ ]:
import joblib
joblib.dump(transformer, processed_dir / 'transformer.joblib')
print('features map:', transformer.get_features_map())
